In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import math
from scipy.stats import spearmanr

In [3]:
file = pd.read_parquet(r"C:\Users\Volodymyr Blonskyi\Chem_AI\Drug-Target\databases\standardised_testing_data.parquet")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pki = torch.tensor(
    file.iloc[:, -2].to_numpy(dtype=np.float32)
)

properties = torch.tensor(
    file.drop(file.columns[-2], axis=1).to_numpy(dtype=np.float32)
)

dataset = torch.utils.data.TensorDataset(properties, pki)

loader = torch.utils.data.DataLoader(dataset, batch_size = 128, shuffle = True, num_workers = 4, pin_memory = True)

In [ ]:
class Model(nn.Module):
  def __init__(self, input_size, layerSize = [2048, 1024, 512, 256]):
    super(Model, self).__init__()
    layers = []
    c = input_size
      
    for s in layerSize:
        layers.append(nn.Linear(c, s))
        layers.append(nn.BatchNorm1d(s))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(0.1))
        c = s
    layers.append(nn.Linear(c, 1))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model(properties.shape[1]).to(device)

weight_trial = int(input("Enter trial number: "))

model.load_state_dict(torch.load(f"model_weights{weight_trial}.pth", map_location=device))
model.eval()

Enter trial number:  2


Model(
  (model): Sequential(
    (0): Linear(in_features=3429, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=512, out_features=256, bias=True)
    (9): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=256, out_features=128, bias=True)
    (13): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.2, inplace=False)
    (16): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [6]:
with torch.no_grad():
    
    avg_MAE = np.array([])
    avg_RMSE = np.array([])
    avg_PPMC = np.array([])
    avg_SC = np.array([])
    for i in range(50):
        for properties, pki in loader:
            
            properties = properties.to(device)
            pki = pki.numpy()
            
            prediction = model(properties).squeeze().cpu()
    
            prediction_array = prediction.numpy()
    
            #Mean absolute error
            MAE = np.mean(np.abs(pki - prediction_array))
    
            #Root mean squared error
            temp = 0
            
            for i in range(len(pki)):
                temp += math.pow(pki[i] - prediction_array[i], 2)
    
            RMSE = math.sqrt( temp / len(pki) )
    
            #Pearson product-moment correlation
            PPMC = np.corrcoef(pki, prediction_array)[0, 1]
    
            #Spearman correlation
            SC = spearmanr(pki, prediction_array).statistic
            
            avg_MAE = np.append(avg_MAE, MAE)
            avg_RMSE = np.append(avg_RMSE, RMSE)
            avg_PPMC = np.append(avg_PPMC, PPMC)
            avg_SC = np.append(avg_SC, SC)

In [7]:
mean_MAE = np.mean(avg_MAE)
mean_RMSE = np.mean(avg_RMSE)
mean_PPMC = np.mean(avg_PPMC)
mean_SC = np.mean(avg_SC)

std_MAE = np.std(avg_MAE)
std_RMSE = np.std(avg_RMSE)
std_PPMC = np.std(avg_PPMC)
std_SC = np.std(avg_SC)

In [22]:
print("Average mean absolute error: ", mean_MAE, "±", std_MAE)
print("Average root mean squared error: ", mean_RMSE, "±", std_RMSE)
print("Average Pearson product-moment correlation: ", mean_PPMC, "±", std_PPMC)
print("Average Spearman correlation: ", mean_SC, "±", std_SC)

Average mean absolute error:  0.48681994365595455 ± 0.04467463182372342
Average root mean squared error:  0.6964381532586471 ± 0.08364723356354484
Average Pearson product-moment correlation:  0.8977255104995542 ± 0.02716935541417497
Average Spearman correlation:  0.8886232972387607 ± 0.026938311888060798


In [4]:
file

,Molecular weight,LogP,Hydrogen bond Donors,Hydrogen bond Acceptors,TPSA,Aromatic Heterocycles,Aromatic Carbocycles,Overall charge,SP3,SP2,...,GearyAuto_Mutability28,GearyAuto_Mutability29,GearyAuto_Mutability30,Mr,Aromaticity,Isoelectric point,Instability index,GRAVY,pKi,Inaccurate
index,,,,,,,,,,,,,,,,,,,,,
1737333604688,0.115308,0.182103,0.204510,-0.346004,-0.121399,-1.014479,0.449093,-0.006029,0.867613,-0.351296,...,-1.878863,0.395812,0.410951,-1.066543,-2.154976,0.698627,0.618006,0.475983,9.096910,0
1737673449536,-0.045933,0.278771,-0.048934,-0.596994,-0.293016,-1.014479,0.449093,-0.006029,0.533455,-0.351296,...,-1.878863,0.395812,0.410951,-1.066543,-2.154976,0.698627,0.618006,0.475983,8.958607,0
0,-0.083188,-0.043940,-0.555821,-1.098975,-0.298615,-1.014479,0.449093,-0.006029,0.422069,-0.192040,...,-1.878863,0.395812,0.410951,-1.066543,-2.154976,0.698627,0.618006,0.475983,8.698970,0
0,0.408276,-0.046558,0.204510,0.155977,0.168220,-1.014479,0.449093,-0.006029,1.090385,-0.192040,...,-1.878863,0.395812,0.410951,-1.066543,-2.154976,0.698627,0.618006,0.475983,8.677781,0
0,0.152008,0.736053,-0.048934,-0.596994,-0.293016,-1.014479,0.449093,-0.006029,0.978999,-0.351296,...,-1.878863,0.395812,0.410951,-1.066543,-2.154976,0.698627,0.618006,0.475983,8.397940,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,1.107903,0.705536,0.711397,1.661919,0.762898,0.940315,1.435918,-0.006029,0.756227,1.320900,...,0.211427,-0.260006,-0.086816,0.145524,0.205251,1.028765,1.280587,-0.307677,9.800001,0
0,1.140958,1.100964,0.711397,1.159939,0.542078,-0.037082,2.422743,-0.006029,0.756227,1.400528,...,0.211427,-0.260006,-0.086816,0.145524,0.205251,1.028765,1.280587,-0.307677,10.099999,0
0,1.250869,1.104324,0.711397,1.410929,0.620378,-0.037082,2.422743,-0.006029,0.867613,1.480156,...,0.211427,-0.260006,-0.086816,0.145524,0.205251,1.028765,1.280587,-0.307677,9.700001,0
